# Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import pykitti
from pathlib import Path
import cv2

# Load KITTI
base = Path.home() / 'SensorTrust' / 'datasets' / 'kitti'
data = pykitti.raw(base_path=str(base), date='2011_09_26', drive='0009')

# Import proxies
from src.proxies.gps_proxy import extract_all_gps_proxies
from src.proxies.imu_proxy import extract_all_imu_proxies
from src.proxies.camera_proxy import extract_all_camera_proxies

print("Setup complete. All imports successful.")

## GPS Proxies

In [ ]:
# Extract GPS proxies
dt = 0.1035  # Your measured frame interval
gps = extract_all_gps_proxies(data.oxts, dt=dt)

# Plot
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8))

ax1.plot(gps['speed'])
ax1.set_ylabel('Speed (m/s)')
ax1.set_title('GPS Forward Speed')
ax1.grid(True, alpha=0.3)

ax2.plot(gps['heading'])
ax2.set_ylabel('Heading (rad)')
ax2.set_title('GPS Heading (bearing)')
ax2.grid(True, alpha=0.3)

ax3.plot(gps['heading_rate'])
ax3.set_ylabel('Heading Rate (rad/s)')
ax3.set_xlabel('Frame')
ax3.set_title('GPS Heading Rate')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Stats
print(f"Speed: mean={np.nanmean(gps['speed']):.2f}, max={np.nanmax(gps['speed']):.2f} m/s")
print(f"Speed NaN: {np.any(np.isnan(gps['speed']))}")
print(f"Heading NaN: {np.any(np.isnan(gps['heading']))} (first frame expected)")
print(f"Heading Rate NaN: {np.any(np.isnan(gps['heading_rate']))} (first 2 frames expected)")

## IMU Proxies

In [ ]:
# Extract IMU proxies
imu = extract_all_imu_proxies(data.oxts, dt=dt)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

ax1.plot(imu['speed'], label='IMU Integrated Speed')
ax1.plot(gps['speed'], label='GPS Speed', alpha=0.7)
ax1.set_ylabel('Speed (m/s)')
ax1.set_title('IMU vs GPS Speed')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(imu['yaw_rate'], label='IMU Yaw Rate')
ax2.plot(gps['heading_rate'], label='GPS Heading Rate', alpha=0.7)
ax2.set_ylabel('Rate (rad/s)')
ax2.set_xlabel('Frame')
ax2.set_title('IMU Yaw Rate vs GPS Heading Rate')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Agreement check
speed_diff = np.abs(imu['speed'] - gps['speed'])
valid = ~np.isnan(speed_diff)
print(f"Speed agreement (mean abs diff): {np.mean(speed_diff[valid]):.3f} m/s")
print(f"Speed agreement (max abs diff):  {np.max(speed_diff[valid]):.3f} m/s")

## CAmera Proxies

In [ ]:
# Load camera frames as numpy arrays
camera_frames = []
for i in range(min(50, len(data.cam2_files))):  # First 50 for speed
    cam_pil = data.get_cam2(i)
    camera_frames.append(np.array(cam_pil))

# Extract camera proxy
cam = extract_all_camera_proxies(camera_frames)

# Plot
plt.figure(figsize=(12, 4))
plt.plot(cam['flow_magnitude'])
plt.xlabel('Frame Pair')
plt.ylabel('Mean Optical Flow (px/frame)')
plt.title('Camera: Mean Optical Flow Magnitude (First 50 Frames)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Flow magnitude: mean={np.mean(cam['flow_magnitude']):.2f}, max={np.max(cam['flow_magnitude']):.2f} px/frame")

## Stationary Check

In [ ]:
# Stationary frames: 404-446
stationary_start = 404
stationary_end = 446

# GPS speed during stationary
stat_speed = gps['speed'][stationary_start:stationary_end]
print("=== STATIONARY CHECK (Frames 404-446) ===")
print(f"GPS speed: mean={np.mean(stat_speed):.4f} m/s, std={np.std(stat_speed):.4f} m/s")
print(f"GPS speed max deviation: {np.max(np.abs(stat_speed)):.4f} m/s")

# IMU speed during stationary
stat_imu_speed = imu['speed'][stationary_start:stationary_end]
print(f"IMU speed: mean={np.mean(stat_imu_speed):.4f} m/s, std={np.std(stat_imu_speed):.4f} m/s")

# IMU yaw rate during stationary
stat_yaw = imu['yaw_rate'][stationary_start:stationary_end]
print(f"IMU yaw: mean={np.mean(stat_yaw):.6f} rad/s, std={np.std(stat_yaw):.6f} rad/s")

print("\nChecks:")
print(f"  GPS stationary: {'PASS' if np.max(np.abs(stat_speed)) < 0.5 else 'FAIL'}")
print(f"  IMU stationary: {'PASS' if np.std(stat_imu_speed) < 0.5 else 'FAIL'}")
print(f"  Yaw stationary: {'PASS' if np.std(stat_yaw) < 0.01 else 'FAIL'}")

## Heading Stability Check

In [ ]:
# Check GPS heading noise at low speed
stat_heading_rate = gps['heading_rate'][stationary_start:stationary_end]
valid_hr = stat_heading_rate[~np.isnan(stat_heading_rate)]

print("=== HEADING STABILITY AT STATIONARY ===")
print(f"Mean heading rate: {np.mean(valid_hr):.6f} rad/s")
print(f"Std heading rate:  {np.std(valid_hr):.6f} rad/s")
print(f"Max heading rate:  {np.max(np.abs(valid_hr)):.6f} rad/s")
print(f"\nHeading gate threshold candidate: {np.std(valid_hr) * 3:.6f} rad/s (3σ)")

## Optical Flow At Rest

In [ ]:
# Camera optical flow on stationary frames
print("=== OPTICAL FLOW AT REST ===")
print("Loading stationary camera frames...")

stat_cam_frames = []
for i in range(stationary_start, min(stationary_end, len(data.cam2_files))):
    stat_cam_frames.append(np.array(data.get_cam2(i)))

stat_cam = extract_all_camera_proxies(stat_cam_frames)

print(f"Stationary flow: mean={np.mean(stat_cam['flow_magnitude']):.4f} px/frame")
print(f"Stationary flow: std={np.std(stat_cam['flow_magnitude']):.4f} px/frame")
print(f"Stationary flow: max={np.max(stat_cam['flow_magnitude']):.4f} px/frame")

# Compare with moving flow (first 50 frames)
moving_flow_mean = np.mean(cam['flow_magnitude'])
print(f"\nMoving flow mean: {moving_flow_mean:.4f} px/frame")
print(f"Ratio moving/stationary: {moving_flow_mean / np.mean(stat_cam['flow_magnitude']):.2f}x")
print(f"Camera usable: {'YES' if moving_flow_mean / np.mean(stat_cam['flow_magnitude']) > 2.0 else 'MARGINAL'}")